# POI Ingestion - Bronze Layer

Downloads OpenStreetMap data from Geofabrik and extracts POI nodes and ways into Unity Catalog.

**Data Source:** Geofabrik OSM Extracts (PBF format)

**Extraction:**
- **Nodes**: Point features (small POIs like ATMs, benches)
- **Ways**: Polygon features (large retail buildings like Walmart, Target)
  - Centroids are calculated from way node coordinates

**Output Table:**
- `{catalog}.{bronze_schema}.raw_pois` - Raw POI data with tags

In [0]:
# MAGIC %md
# MAGIC ## Parameters

In [0]:
import requests
import shutil
import os
import yaml
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import uuid

# Notebook parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("osm_url", "https://download.geofabrik.de/north-america/us/massachusetts-latest.osm.pbf")
dbutils.widgets.text("osm_region", "massachusetts")
dbutils.widgets.text("config_path", "")

# Extract parameters
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
osm_url = dbutils.widgets.get("osm_url")
osm_region = dbutils.widgets.get("osm_region")
config_path = dbutils.widgets.get("config_path")

assert catalog and bronze_schema and osm_url, "Missing required parameters: catalog, bronze_schema, osm_url"

# Define paths
osm_filename = osm_url.split('/')[-1]
osm_volume_path = f"/Volumes/{catalog}/{bronze_schema}/osm_data/"
osm_file_path = f"{osm_volume_path}{osm_filename}"
output_table = f"{catalog}.{bronze_schema}.raw_pois"

print(f"Catalog: {catalog}")
print(f"Schema: {bronze_schema}")
print(f"OSM URL: {osm_url}")
print(f"Output table: {output_table}")

In [0]:
# MAGIC %md
# MAGIC ## Section 1: Download OSM Data

In [ ]:
download_id = str(uuid.uuid4())
download_start = datetime.now()

# Check if file already exists (idempotency)
if os.path.exists(osm_file_path):
    file_size_mb = os.path.getsize(osm_file_path) / (1024 * 1024)
    status = "existing"
    print(f"File already exists: {osm_file_path} ({file_size_mb:.2f} MB)")
else:
    print(f"Downloading {osm_url}...")
    
    # Verify volume exists by trying to list it (must include volume name)
    try:
        # Use dbutils to check if volume is accessible
        dbutils.fs.ls(f"/Volumes/{catalog}/{bronze_schema}/osm_data/")
        print(f"✓ Volume path is accessible: {osm_volume_path}")
    except Exception as e:
        print(f"\n❌ ERROR: Unity Catalog volume does not exist or is not accessible")
        print(f"\nExpected volume: {catalog}.{bronze_schema}.osm_data")
        print(f"\nPlease verify:")
        print(f"  1. Bundle was deployed: databricks bundle deploy")
        print(f"  2. Volume exists in Unity Catalog")
        print(f"  3. You have permissions to access the volume")
        print(f"\nYou can check with: databricks volumes list {catalog}.{bronze_schema}")
        raise RuntimeError(f"Volume not accessible: {osm_volume_path}") from e
    
    # Download directly to volume (no need to create directory - volume already exists)
    with requests.get(osm_url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(osm_file_path, "wb") as f:
            shutil.copyfileobj(r.raw, f)
    
    file_size_mb = os.path.getsize(osm_file_path) / (1024 * 1024)
    download_end = datetime.now()
    duration_seconds = (download_end - download_start).total_seconds()
    status = "completed"
    print(f"Download completed: {file_size_mb:.2f} MB in {duration_seconds:.1f} seconds")

# Verify file exists
if not os.path.exists(osm_file_path):
    raise RuntimeError(f"OSM file not found at: {osm_file_path}")

print(f"\nOSM file ready: {osm_file_path}")

In [0]:
# MAGIC %md
# MAGIC ## Section 2: Extract POIs

In [ ]:
import osmium

# Load configuration if provided
extract_all = False
poi_tag_categories = ['amenity', 'shop']

if config_path and os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    poi_config = config.get('poi_extraction', {})
    extract_all = poi_config.get('extract_all', True)
    poi_tag_categories = poi_config.get('poi_tag_categories', poi_tag_categories)
    print(f"Loaded config: extract_all={extract_all}, categories={poi_tag_categories}")
else:
    print(f"Using default POI extraction settings")

# OPTIMIZATION: Filter by brand name during extraction
# Only extract POIs that match partner/competitor brand patterns
# General POI counts come from CARTO Marketplace (h3_features_clean.total_poi_count)
BRAND_FILTER_ENABLED = True  # Set False to extract all POIs (slow)

PARTNER_BRANDS = [
    'walmart', 'walmart supercenter', 'speedway', '7-eleven', '7 eleven', "shaw's", 'shaws'
]
COMPETITOR_BRANDS = [
    'pizza hut', "domino's", 'dominos', "papa john's", 'papa johns'
]

ALL_BRAND_PATTERNS = PARTNER_BRANDS + COMPETITOR_BRANDS
print(f"\nBrand filter enabled: {BRAND_FILTER_ENABLED}")
if BRAND_FILTER_ENABLED:
    print(f"Partner brands: {PARTNER_BRANDS}")
    print(f"Competitor brands: {COMPETITOR_BRANDS}")
    print(f"\nNote: General POI counts from CARTO Marketplace (not OSM extraction)")

In [ ]:
class POIHandler(osmium.SimpleHandler):
    """Handler to extract POI nodes and ways from OSM data based on tag categories.
    
    Optionally filters by brand names and POI subcategories to reduce extraction volume.
    """
    
    def __init__(self, extract_all=True, poi_tag_categories=None, 
                 brand_filter_enabled=False, brand_patterns=None, subcategory_filter=None):
        super().__init__()
        self.pois = []
        self.skipped_count = 0  # Track filtered out POIs
        
        default_poi_tags = ['amenity', 'shop']
        
        if extract_all:
            self.poi_tag_categories = default_poi_tags
        else:
            self.poi_tag_categories = poi_tag_categories if poi_tag_categories else default_poi_tags
        
        # Brand/subcategory filtering
        self.brand_filter_enabled = brand_filter_enabled
        self.brand_patterns = [b.lower() for b in (brand_patterns or [])]
        self.subcategory_filter = set(s.lower() for s in (subcategory_filter or []))
    
    def _has_poi_tag(self, tags):
        """Check if element has any POI tag from the configured categories"""
        tag_keys = {tag.k for tag in tags}
        return any(poi_tag in tag_keys for poi_tag in self.poi_tag_categories)
    
    def _matches_filter(self, tags_dict):
        """Check if POI matches brand patterns or subcategory filter.
        
        Returns True if:
        - Brand filtering is disabled (extract all)
        - POI name matches any brand pattern (case-insensitive partial match)
        - POI amenity/shop tag matches subcategory filter
        """
        if not self.brand_filter_enabled:
            return True
        
        # Check brand name match
        name = tags_dict.get('name', '').lower()
        if name:
            for pattern in self.brand_patterns:
                if pattern in name:
                    return True
        
        # Check subcategory match (amenity or shop value)
        amenity = tags_dict.get('amenity', '').lower()
        shop = tags_dict.get('shop', '').lower()
        
        if amenity in self.subcategory_filter:
            return True
        if shop in self.subcategory_filter:
            return True
        
        return False
    
    def node(self, n):
        """Extract nodes with POI tags"""
        if not n.location.valid():
            return
        
        if self._has_poi_tag(n.tags):
            tags_dict = dict(n.tags)
            if tags_dict:
                # Apply brand/subcategory filter
                if not self._matches_filter(tags_dict):
                    self.skipped_count += 1
                    return
                
                self.pois.append({
                    'osm_id': str(n.id),
                    'osm_type': 'node',
                    'latitude': n.location.lat,
                    'longitude': n.location.lon,
                    'tags': tags_dict
                })
    
    def way(self, w):
        """Extract ways with POI tags (e.g., large retail buildings like Walmart)
        
        Ways are polygons/lines defined by a sequence of nodes. We calculate the
        centroid to get a single lat/lon point for the POI.
        """
        if not self._has_poi_tag(w.tags):
            return
        
        tags_dict = dict(w.tags)
        if not tags_dict:
            return
        
        # Apply brand/subcategory filter
        if not self._matches_filter(tags_dict):
            self.skipped_count += 1
            return
        
        # Collect valid node locations to calculate centroid
        lats = []
        lons = []
        for node in w.nodes:
            if node.location.valid():
                lats.append(node.location.lat)
                lons.append(node.location.lon)
        
        # Only add if we have valid coordinates
        if lats and lons:
            self.pois.append({
                'osm_id': str(w.id),
                'osm_type': 'way',
                'latitude': sum(lats) / len(lats),  # Centroid latitude
                'longitude': sum(lons) / len(lons),  # Centroid longitude
                'tags': tags_dict
            })

In [ ]:
# Parse OSM file and extract POIs (nodes and ways)
print(f"Extracting POIs from {osm_file_path}...")
print("Extracting both nodes (points) and ways (building polygons)...")

if BRAND_FILTER_ENABLED:
    print(f"Brand filtering ENABLED - extracting only partner/competitor brands")
else:
    print(f"Brand filtering DISABLED - extracting ALL POIs (slow)")

handler = POIHandler(
    extract_all=extract_all, 
    poi_tag_categories=poi_tag_categories,
    brand_filter_enabled=BRAND_FILTER_ENABLED,
    brand_patterns=ALL_BRAND_PATTERNS,
    subcategory_filter=None  # General POI counts come from CARTO, not OSM
)

# Use locations=True to resolve node coordinates for ways
# This allows the way() handler to access node.location for centroid calculation
handler.apply_file(osm_file_path, locations=True)

poi_count = len(handler.pois)
node_count = sum(1 for p in handler.pois if p['osm_type'] == 'node')
way_count = sum(1 for p in handler.pois if p['osm_type'] == 'way')

print(f"\nExtracted {poi_count:,} total POIs:")
print(f"  - Nodes: {node_count:,}")
print(f"  - Ways:  {way_count:,}")

if BRAND_FILTER_ENABLED:
    print(f"  - Skipped (filtered out): {handler.skipped_count:,}")
    reduction_pct = (handler.skipped_count / (poi_count + handler.skipped_count) * 100) if (poi_count + handler.skipped_count) > 0 else 0
    print(f"  - Reduction: {reduction_pct:.1f}%")

if poi_count == 0:
    raise RuntimeError("No POIs found in OSM file. Check if file contains POI data with matching tags.")

In [0]:
# MAGIC %md
# MAGIC ## Section 3: Write to Bronze Table

In [0]:
# Convert POIs to Spark DataFrame
schema = StructType([
    StructField("osm_id", StringType(), False),
    StructField("osm_type", StringType(), False),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("tags", MapType(StringType(), StringType()), True)
])

poi_df = spark.createDataFrame(handler.pois, schema=schema)

# Add ingestion metadata
poi_df = poi_df.withColumn("ingestion_timestamp", F.current_timestamp())

print(f"Created DataFrame with {poi_df.count():,} rows")
display(poi_df.limit(5))

In [0]:
# Write to Bronze table
(poi_df
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .option("delta.autoOptimize.optimizeWrite", "true")
 .saveAsTable(output_table))

print(f"Written to {output_table}")

In [0]:
# MAGIC %md
# MAGIC ## Validation

In [ ]:
print("=" * 80)
print("POI INGESTION VALIDATION")
print("=" * 80)

summary = spark.sql(f"""
    SELECT 
        COUNT(*) as total_pois,
        COUNT(DISTINCT osm_id) as unique_pois,
        COUNT(CASE WHEN latitude IS NOT NULL AND longitude IS NOT NULL THEN 1 END) as pois_with_coords,
        COUNT(CASE WHEN tags['name'] IS NOT NULL THEN 1 END) as pois_with_name,
        COUNT(CASE WHEN tags['addr:street'] IS NOT NULL THEN 1 END) as pois_with_address
    FROM {output_table}
""")
display(summary)

# Show all extracted branded POIs (should only be partners and competitors)
print("\nExtracted branded POIs:")
spark.sql(f"""
    SELECT 
        tags['name'] as name,
        COALESCE(tags['amenity'], tags['shop']) as category,
        osm_type,
        COUNT(*) as count
    FROM {output_table}
    WHERE tags['name'] IS NOT NULL
    GROUP BY tags['name'], COALESCE(tags['amenity'], tags['shop']), osm_type
    ORDER BY count DESC
""").show(50, truncate=False)

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)